# Experiment 39 - Exact Value CatBoost + 3-Seed Average

Goal: build on the strongest part of the previous search while testing a genuinely different representation.

This experiment treats selected discrete numeric values as identities by passing string copies to CatBoost as categorical features. It keeps the existing numeric values and the strongest subsidy interaction features, then averages three CPU CatBoost seeds. The run is intentionally capped at the three seeds so the expected runtime stays under about 4 hours.

**Benchmark:** Exp 33B = 0.945331 OOF ROC-AUC.

**Public S6E9 evidence:** exact numeric value identity as CatBoost categoricals produced about +0.00337 OOF over a CatBoost baseline in one public reproduction, and a separate public run reported 0.94550 OOF / 0.94570 LB with seed averaging. This experiment tests the same core representation locally, without copying their external-source dependency.


In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

TRAIN_PATH = '../data/train.csv'
TARGET = 'Will_Buy_EV'

train = pd.read_csv(TRAIN_PATH)
y = train[TARGET].astype(str).str.strip().map({'No': 0, 'Yes': 1}).astype(int)

X = train.drop(columns=[TARGET]).copy()
X = X.drop(columns=['id'])

numeric_cols = [
    'Age', 'Annual_Income_USD', 'Daily_Commute_km',
    'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work', 'Environmental_Concern_Level'
]

categorical_cols = [
    'Gender', 'City_Type', 'Current_Car_Type',
    'Home_Charging_Possible', 'Subsidy_Available',
    'Range_Anxiety_Level'
]

# Strongest cheap interaction family used earlier in the project.
sub = X['Subsidy_Available'].astype('string').fillna('__MISSING__').eq('Yes').astype(int)
home = X['Home_Charging_Possible'].astype('string').fillna('__MISSING__').eq('Yes').astype(int)
anxiety = pd.to_numeric(X['Range_Anxiety_Level'], errors='coerce')

X['Subsidy_x_EnvConcern'] = sub * X['Environmental_Concern_Level']
X['Subsidy_x_Income'] = sub * X['Annual_Income_USD']
X['Subsidy_x_HomeCharging'] = sub * home

# Exact numeric identities. Keep the original numeric columns too.
value_identity_cols = [
    'Age',
    'Annual_Income_USD',
    'Daily_Commute_km',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work'
]

for col in value_identity_cols:
    X[f'{col}__value_id'] = (
        X[col].astype('string').fillna('__MISSING__')
    )

cat_identity_cols = categorical_cols + [f'{c}__value_id' for c in value_identity_cols]

for col in cat_identity_cols:
    X[col] = X[col].astype('string').fillna('__MISSING__')

print('Rows:', len(X))
print('Categorical columns:', len(cat_identity_cols))
print('Value identity columns:', [f'{c}__value_id' for c in value_identity_cols])


Rows: 668665
Categorical columns: 11
Value identity columns: ['Age__value_id', 'Annual_Income_USD__value_id', 'Daily_Commute_km__value_id', 'Charging_Stations_Near_Home__value_id', 'Charging_Stations_Near_Work__value_id']


In [4]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
seeds = [42, 7, 2026]

all_seed_oof = []
fold_scores = {}

def run_seed(seed):
    oof = np.zeros(len(X), dtype=float)
    fold_auc = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        X_tr = X.iloc[tr_idx].copy()
        X_va = X.iloc[va_idx].copy()
        y_tr = y.iloc[tr_idx]
        y_va = y.iloc[va_idx]

        model = CatBoostClassifier(
            iterations=2000,
            learning_rate=0.05,
            depth=6,
            loss_function='Logloss',
            eval_metric='AUC',
            l2_leaf_reg=3,
            random_strength=1,
            bootstrap_type='Bayesian',
            bagging_temperature=1,
            random_seed=seed,
            thread_count=-1,
            verbose=False,
            allow_writing_files=False
        )

        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_identity_cols,
            eval_set=(X_va, y_va),
            use_best_model=True,
            verbose=False
        )

        pred = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = pred
        auc = roc_auc_score(y_va, pred)
        fold_auc.append(auc)
        print(f'Seed {seed} | Fold {fold}/5 | AUC: {auc:.6f}')

    overall = roc_auc_score(y, oof)
    print(f'Seed {seed} OOF AUC: {overall:.6f}')
    return oof, fold_auc

for seed in seeds:
    oof_seed, scores = run_seed(seed)
    all_seed_oof.append(oof_seed)
    fold_scores[seed] = scores

oof_avg3 = np.mean(np.vstack(all_seed_oof), axis=0)
score = roc_auc_score(y, oof_avg3)

print('\n' + '=' * 70)
print('EXPERIMENT 39 RESULT')
print('=' * 70)
print(f'39A_Exact_Value_CatBoost_Avg3 ROC-AUC: {score:.6f}')
print('33B benchmark: 0.945331')
print(f'vs 33B: {score - 0.945331:+.6f}')
print('0.950 target: 0.950000')
print(f'vs 0.950: {score - 0.950000:+.6f}')
print('0.960 target: 0.960000')
print(f'vs 0.960: {score - 0.960000:+.6f}')


Seed 42 | Fold 1/5 | AUC: 0.944422
Seed 42 | Fold 2/5 | AUC: 0.945068
Seed 42 | Fold 3/5 | AUC: 0.946350
Seed 42 | Fold 4/5 | AUC: 0.945643
Seed 42 | Fold 5/5 | AUC: 0.945377
Seed 42 OOF AUC: 0.945367
Seed 7 | Fold 1/5 | AUC: 0.944431
Seed 7 | Fold 2/5 | AUC: 0.944978
Seed 7 | Fold 3/5 | AUC: 0.946433
Seed 7 | Fold 4/5 | AUC: 0.945627
Seed 7 | Fold 5/5 | AUC: 0.945427
Seed 7 OOF AUC: 0.945370
Seed 2026 | Fold 1/5 | AUC: 0.944476
Seed 2026 | Fold 2/5 | AUC: 0.945015
Seed 2026 | Fold 3/5 | AUC: 0.946328
Seed 2026 | Fold 4/5 | AUC: 0.945676
Seed 2026 | Fold 5/5 | AUC: 0.945352
Seed 2026 OOF AUC: 0.945362

EXPERIMENT 39 RESULT
39A_Exact_Value_CatBoost_Avg3 ROC-AUC: 0.945426
33B benchmark: 0.945331
vs 33B: +0.000095
0.950 target: 0.950000
vs 0.950: -0.004574
0.960 target: 0.960000
vs 0.960: -0.014574
